# Update static data

Loads the legacy data sources (courses, students, study programme stats), cleans and normalises them, and writes them to the `static_data` schema in Postgres.

Tables written (each one is **replaced** on every run):

| Table | Source |
|---|---|
| `static_data.courses` | `old_data_sources/coursesDB.json` |
| `static_data.students` | `old_data_sources/studentsDB.csv`; `get_student_data.ipynb` writes the same table from AIT's live view |
| `static_data.study_programs` | `old_data_sources/studyProgramStatsDB.xlsx` |

In [ ]:
from pis_lib.connect import get_pg_engine
import pandas as pd

## Connect to the database

In [ ]:
pg = get_pg_engine()

## Load raw sources

All files live in `old_data_sources/`. Course numbers are the JSON keys, so `convert_axes=False` keeps them as strings (e.g. `"01001"` keeps its leading zero).

In [ ]:
raw_courses_df = pd.read_json("old_data_sources/coursesDB.json", orient="index",  convert_axes=False)
raw_students_df = pd.read_csv("old_data_sources/studentsDB.csv", sep=",")
raw_study_programs_df = pd.read_excel("old_data_sources/studyProgramStatsDB.xlsx", sheet_name="Sheet1")
raw_student_admissions_df = pd.read_excel("old_data_sources/DTU Student Admission 2022-2025 MF.xlsx", sheet_name="Sheet1")

## Courses

Keep only the course name and turn the index (course number) into a column.

In [ ]:
courses_df = (
    raw_courses_df[["name"]]
    .rename(columns={"name": "course_name"})
    .rename_axis("course_number")
    .reset_index()
)

## Students

Steps:

1. **Rename** and **Reorder** the source columns to English names `COLUMN_NAMES` and in order `COLUMN_ORDER`.
2. **Gender**: map `M`/`K` to `Male`/`Female`; anything else becomes `Other`.
3. **Education**: normalise `uddannelse_dk` (lowercase, dots removed) and translate it with `STUDENT_EDU_MAP`. Rows with a missing or `-` code fall back to `EDU_FALLBACK`, looked up by direction. Anything still unmatched becomes `Other`.
4. **Direction**: collapse spelling variants and former programme names in `retning_dk` into one label each (`DIRECTION_ALIASES`).


In [ ]:
COLUMN_NAMES = {
    "KoenFraCpr"       : "gender",
    "stads_studentcode": "student_number",
    "retning_uk"       : "direction_EN",
    "retning_dk"       : "direction_DK",
    "uddannelse_dk"    : "education_DK",
}

COLUMN_ORDER = [
    "student_number",
    "gender",
    "education_EN",
    "education_DK",
    "direction_DK",
    "direction_EN",
]

GENDER_MAP = {"M": "Male", "K": "Female"}

  # keys are uddannelse_dk lowercased with dots removed ("ph.d" -> "phd", "åben udd." -> "åben udd")
STUDENT_EDU_MAP = {
    "bachelor"     : "BSc in Engineering",
    "kandidat"     : "MSc in Engineering",
    "diploming"    : "Diplom / BEng",
    "eksporting"   : "Diplom / BEng",
    "phd"          : "PhD",
    "deltidsdiplom": "Part-time diploma",
    "master"       : "Part-time master",
    "åben udd"     : "Open education",
    "enkeltfag"    : "Single course",
    "adgangskursus": "Access course",
    "gæst udl"     : "Exchange student",
    "gæst dk"      : "Guest student",
    "talent"       : "Talent programme",
}

  # used only for rows whose education code is "-" or missing
EDU_FALLBACK = {
    "Foreign exchange student": "Exchange student",
    "Deltidsdiplom"           : "Part-time diploma",
    "Tek. entreprenørskab"    : "MSc in Engineering",
}

# spelling variants in retning_dk -> one label per programme
DIRECTION_ALIASES = {
    "Medicin og tek"            : "Medicin og tek.",
    "Life science"              : "Life Science",
    "Miljøteknologi"            : "Miljøtek.",
    "Cyberteknologi"            : "Cybertek.",
    "Netværksteknologi"         : "Cybertek.",                          # former name
    "Mat. og tek."              : "Matematik og tek",
    "Data Science og Management": "Data science og management",
    "Bæredygtig energidesign"   : "Bæredygtigt energidesign",
    "Tek. entrepreneurskab"     : "Tek. entreprenørskab",
    "Farmateknologi"            : "Farmatek.",
    "Autonome systemer"         : "Autonome sys.",
    "Systembiologi"             : "Bioinfo. og systembio.",
    "Kemisk/biokemisk tek."     : "Kemisk og biokemisk tek.",
    "Biotek"                    : "Bioteknologi",
    "Industriel økonomi"        : "Industriel øko. og tekn. ledelse",
    "Mat. modellering"          : "Mat. model. og computing",
    "Konst. og mekanik"         : "Konstruktion og mek.",
    "Akustisk tek."             : "Lyd og akustisk tek.",
    "Telekom."                  : "Kom.tek. og system",
    "Digitale medietek."        : "Menneskeorienteret AI",              # former name
    "Transp. og logistik"       : "Transport og logistik",
    "Materialetek."             : "Materiale og procestek.",
    "Fødevareteknologi"         : "Fødevaretek.",
    "Maskin"                    : "Maskinteknik",
    "Kemi- og biotek."          : "Kemi/biotek.",
    "Vand, bioress. og miljø"   : "Miljøtek.",
    "Trafik og transport"       : "Mobilitet/transport",
    "IT elektronik"             : "Computer eng.",                      # retning_uk is "Computer Engineering"
    "Elektro"                   : "Elektrotek.",
}



students_df = (
    raw_students_df
    .drop(columns=["displayName", "study_frame", "study_frame2", "uddannelse_uk"])
    .rename(columns=COLUMN_NAMES, errors="raise")
    .assign(
        gender=lambda d: d["gender"].map(GENDER_MAP).fillna("Other"),
        education_DK=lambda d: (
            d["education_DK"].str.lower().str.replace(".", "", regex=False).str.strip()
        ),
        education_EN=lambda d: (
            d["education_DK"].map(STUDENT_EDU_MAP)
            .fillna(d["direction_DK"].fillna(d["direction_EN"]).map(EDU_FALLBACK))
            .fillna("Other")
        ),
        direction_DK=lambda d: d["direction_DK"].str.strip().replace(DIRECTION_ALIASES),
    )
    [COLUMN_ORDER]
)

## Study programmes

Yearly enrolment counts per education and direction.

- **Education** is mapped to the same labels as `students.education_EN` (`PROGRAM_EDU_MAP`, applied as a regex so partial matches are replaced).
- **Direction** is mapped to the same short Danish labels as `students.direction_DK` (`PROGRAM_DIRECTION_MAP`), so the two tables can be joined.
- Year columns use `-` and `*` for missing or suppressed counts. Both become `0`, then the columns are cast to `int64`.

In [ ]:
PROGRAM_EDU_MAP = {
    "Civilbachelor" : "BSc in Engineering",
    "Diplomingeniør": "Diplom / BEng",
    "Kandidat"      : "MSc in Engineering",
}

PROGRAM_DIRECTION_MAP = {
    "Computer Engineering"                        : "Computer eng.",
    "Kemi og Teknologi"                           : "Kemi og tek.",
    "Arktisk Byggeri og Infrastruktur"            : "Arktisk tek.",
    "Byggeri og Infrastruktur"                    : "Byggeri og infrast.",
    "Elektrisk Energiteknologi"                   : "Elektrisk energitek.",
    "Elektroteknologi"                            : "Elektrotek.",
    "Fødevaresikkerhed og -kvalitet"              : "Fødevareanaly.",
    "Global Business og Teknologi"                : "Global Biz og tek.",
    "Kemi- og Bioteknik"                          : "Kemi/biotek.",
    "Kemiteknik og International Business"        : "Kemitek./int. business",
    "Maritim Teknik"                              : "Maritim tek.",
    "Mobilitet, Transport og Logistik"            : "Mobilitet/transport",
    "Softwareteknologi"                           : "Softwaretek.",
    "Sundhedsteknologi"                           : "Sundhedstek.",
    "Anvendt Matematik"                           : "Matematik og tek",
    "Byggeteknologi"                              : "Byggetek.",
    "Cyberteknologi"                              : "Cybertek.",
    "Design og Innovation"                        : "Design og inn.",
    "Fysik og Ingeniørvidenskab"                  : "Fysik og ing.videnskab",
    "Geofysik og Rumteknologi"                    : "Geofysik og rumtek.",
    "Kvan. Biologi og Sygdomsmodel. (2020 og før)": "Kvantitativ biologi",
    "Life Science og Teknologi"                   : "Life Science",
    "Miljøteknologi"                              : "Miljøtek.",
    "Autonome Systemer"                           : "Autonome sys.",
    "Bioinformatik"                               : "Bioinfo. og systembio.",
    "Bæredygtig Energi"                           : "Bæredygtig energi",
    "Bæredygtige Energisystemer"                  : "Bæredygtig energisys.",
    "Bæredygtige Energiteknologier"               : "Bæredygtige energitek.",
    "Bæredygtigt Fiskeri og Akvakultur"           : "Bæredygtigt fisk. og akva.",
    "Biomaterialeteknologier til Medicin"         : "Biomaterialeteknologier til med.",
    "Farmateknologi"                              : "Farmatek.",
    "Fysik og Teknologi"                          : "Fysik og tek.",
    "Fødevareteknologi"                           : "Fødevaretek.",
    "Informationsteknologi"                       : "Informationstek.",
    "Kommunikationsteknologier og Systemdesign"   : "Kom.tek. og system",
    "Konstruktion og Mekanik"                     : "Konstruktion og mek.",
    "Lys og Teknologi"                            : "Lys og tek.",
    "Matematisk Modellering og Computing"         : "Mat. model. og computing",
    "Materiale- og Procesteknologi"               : "Materiale og procestek.",
    "Medicin og tek"                              : "Medicin og tek.",
    "Menneskeorienteret Kunstig Intelligens"      : "Menneskeorienteret AI",
    "Teknologisk Entreprenørskab"                 : "Tek. entreprenørskab",
    "Medicin og Teknologi"                        : "Medicin og tek.",
    "Industriel Økonomi og Teknologiledelse"      : "Industriel øko. og tekn. ledelse",
    "Kemisk og Biokemisk Teknologi"               : "Kemisk og biokemisk tek.",
    "Kvantitativ Biologi og Sygdomsmodellering"   : "Kvant. biologi og sygdomsmod.",
    "Lyd og Akustisk Teknologi"                   : "Lyd og akustisk tek.",
    "Anvendt Kemi"                                : "Anvendt kemi",
    "Business Analytics"                          : "Business analytics",
    "Bæredygtigt Energidesign"                    : "Bæredygtigt energidesign",
    "Data Science og Management"                  : "Data science og management",
    "General Engineering"                         : "General engineering",
    "IT og Økonomi"                               : "IT og økonomi",
    "Kunstig Intelligens og Data"                 : "Kunstig intelligens og data",
    "Marin Ingeniørvidenskab"                     : "Marin ingeniørvidenskab",
    "Proces og Innovation"                        : "Proces og innovation",
    "Teknisk Biomedicin"                          : "Teknisk biomedicin",
    "Transport og Logistik"                       : "Transport og logistik",
    "Olie- og Gasteknologi"                       : "Olie- og gastek.",
    "Skibsteknik og Maritimt Design"              : "Skibsteknik",
    "Mekanisk Design og Teknologi"                : "Prod. og konst.",
}

ID_COLS = ["Education", "Direction"]
year_cols = raw_study_programs_df.columns.drop(ID_COLS)

study_programs_df = (
    raw_study_programs_df
    .assign(
        Education=lambda d: d["Education"].replace(PROGRAM_EDU_MAP, regex=True),
        Direction=lambda d: d["Direction"].replace(PROGRAM_DIRECTION_MAP),
    )
    .replace({col: {"-": "0", "*": "0"} for col in year_cols})
    .astype(dict.fromkeys(year_cols, "int64"))
)

## Write to `static_data`

> **Warning:** `if_exists="replace"` drops and recreates each table, so the old contents are lost on every run.

In [ ]:
courses_df.to_sql("courses", pg, schema="static_data", if_exists="replace", index=False)
students_df.to_sql("students", pg, schema="static_data", if_exists="replace", index=False)
study_programs_df.to_sql("study_programs", pg, schema="static_data", if_exists="replace", index=False)